| # |  |  |
|---|----------|---------------|
| 1 | Regras de matching semânticas no prompt | Peeters & Bizer (2023) — regras em linguagem natural alcançam +5.94% F1 com custo mínimo |
| 3 | Pré-normalização do título antes do LLM | Ditto (Li et al., 2020) — span normalization reduz carga cognitiva do modelo |
| 4 | Batch prompting (múltiplos títulos por chamada) | BATCHER (Fan et al., 2024) — 4-7× redução de custo com qualidade equivalente |

**Arquitetura em 5 passos:**
1. Ingestão e Amostragem
2. Pré-Normalização Simbólica do Título (NOVO — Ditto-inspired)
3. Extração via LLM com Batch Prompting (prompt melhorado)
4. Limpeza Determinística (Camada Simbólica)
5. Exportação + Avaliação Estruturada

## Setup

In [1]:
!pip install -q tqdm google-generativeai python-dotenv


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import re
import json
import time
import logging
import pandas as pd
import numpy as np
from tqdm import tqdm
from pathlib import Path
from dotenv import load_dotenv
import google.generativeai as genai
import sys

load_dotenv()

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

c:\Users\nunes\.pyenv\pyenv-win\versions\3.12.3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ── Configuração ──────────────────────────────────────────────
INPUT_PATH      = "data/train.csv"          # CSV de entrada (Stage 1)
INPUT_COLUMN    = "title"                  # Coluna com os títulos originais
ATTACK_PATH     = "output/attack.csv"      # Saída do Stage 1
NORMALIZED_PATH = "output/normalized.csv"  # Saída do Stage 2

RANDOM_SEED  = 42
BATCH_SIZE   = 5
API_KEY      = os.getenv("GOOGLE_API_KEY")
LLM_MODEL    = "gemini-3.1-flash-lite-preview"
TEMPERATURE  = 0.0

## Passo 1 — Ingestão

In [4]:
df = pd.read_csv(INPUT_PATH)
print(f"Total de títulos: {len(df)}")
df.head(5)

Total de títulos: 20000000


,title,label_quality,language,category
0,Hidrolavadora Lavor One 120 Bar 1700w Bomba A...,unreliable,spanish,ELECTRIC_PRESSURE_WASHERS
1,Placa De Sonido - Behringer Umc22,unreliable,spanish,SOUND_CARDS
2,Maquina De Lavar Electrolux 12 Kilos,unreliable,portuguese,WASHING_MACHINES
3,Par Disco De Freio Diant Vent Gol 8v 08/ Frema...,unreliable,portuguese,VEHICLE_BRAKE_DISCS
4,Flashes Led Pestañas Luminoso Falso Pestañas P...,unreliable,spanish,FALSE_EYELASHES


In [5]:
df_sample = df[[INPUT_COLUMN]].copy().reset_index(drop=True)
print(f"Títulos para processar: {len(df_sample)}")

Títulos para processar: 20000000


In [6]:
df_sample = df[[INPUT_COLUMN]].sample(n=100, random_state=RANDOM_SEED).reset_index(drop=True)
print(f"Títulos para processar: {len(df_sample)}")

Títulos para processar: 100


## Passo 2 — Attack Augmentation via LLM

O LLM gera variantes adversariais dos títulos originais para testar robustez do normalizador.

In [7]:
ATTACK_PROMPT = """
Você é um gerador de variantes adversariais de títulos de produtos de e-commerce brasileiro.

TAREFA
Para cada título recebido (identificado por idx), gere 6 (SEIS) variantes adversariais, distribuídas em 3 estratégias diferentes.
Retorne SOMENTE um array JSON. Sem texto extra, sem markdown.

O formato OBRIGATÓRIO do JSON para CADA objeto no array deve ser:
{
  "idx": int,
  "attacks": [
    {"tipo": "Abreviacao", "texto": "var1"},
    {"tipo": "Abreviacao", "texto": "var2"},
    {"tipo": "Notacao_Unidades", "texto": "var3"},
    {"tipo": "Notacao_Unidades", "texto": "var4"},
    {"tipo": "Erro_Digitacao", "texto": "var5"},
    {"tipo": "Erro_Digitacao", "texto": "var6"}
  ]
}

ESTRATÉGIAS OBRIGATÓRIAS (Faça exatamente 2 variantes diferentes para cada tipo):
  1. "Abreviacao": Abreviar palavras longas (ex: "Liquidificador" → "Liq.")
  2. "Notacao_Unidades": Notação alternativa de unidades (ex: "500ml" → "0,5L", "12kg" → "12 kilos")
  3. "Erro_Digitacao": Erro de digitação / Trocar caracteres adjacentes (ex: "Samsung" → "Samsugn")
"""

In [8]:
def attack_batch_llm(
    titles: list[str],
    client: genai.GenerativeModel,
    start_idx: int = 0,
) -> list[dict]:
    lines = [f'[{start_idx + i}] "{t}"' for i, t in enumerate(titles)]
    user_prompt = "\n".join(lines)

    try:
        response = client.generate_content(user_prompt)
        raw_text = response.text.strip()
        if raw_text.startswith("```"):
            raw_text = re.sub(r"^```(?:json)?", "", raw_text).rstrip("`").strip()
        parsed = json.loads(raw_text)
        if isinstance(parsed, dict):
            parsed = [parsed]
        return parsed
    except json.JSONDecodeError:
        matches = re.findall(r"\{[^{}]+\}", raw_text, re.DOTALL)
        results = []
        for m in matches:
            try:
                results.append(json.loads(m))
            except json.JSONDecodeError:
                continue
        return results or [{"idx": start_idx + i, "attacked": t} for i, t in enumerate(titles)]
    except Exception as e:
        logger.error("Erro attack LLM: %s", e)
        return [{"idx": start_idx + i, "attacked": t} for i, t in enumerate(titles)]

In [9]:
os.makedirs("output", exist_ok=True)

genai.configure(api_key=API_KEY)
attack_client = genai.GenerativeModel(
    model_name=LLM_MODEL,
    system_instruction=ATTACK_PROMPT,
    generation_config=genai.GenerationConfig(
        response_mime_type="application/json",
        temperature=0.7,  
        max_output_tokens=2048,
    ),
)

titles_orig = df_sample[INPUT_COLUMN].tolist()
n_attack = len(titles_orig)

registros_ataque = []

for batch_start in tqdm(range(0, n_attack, BATCH_SIZE), desc="Attack", unit="batch"):
    batch_end = min(batch_start + BATCH_SIZE, n_attack)
    batch = titles_orig[batch_start:batch_end]

    # Substitua pelo nome correto da sua função/cliente
    raw_list = attack_batch_llm(batch, attack_client, start_idx=batch_start)

    for local_i in range(batch_end - batch_start):
        global_i = batch_start + local_i
        
        # Procura a resposta correspondente deste idx do batch
        matched = next(
            (r for r in raw_list if isinstance(r, dict) and r.get("idx") == global_i),
            None,
        )
        
        # Confere se a IA gerou a lista de dicionários "attacks" corretamente
        if matched and "attacks" in matched and isinstance(matched["attacks"], list):
            ataques = matched["attacks"]
        else:
            # Fallback caso a IA falhe ou gagueje feio
            ataques = [{"tipo": "Fallback_Erro_LLM", "texto": titles_orig[global_i]}] * 6
            
        for ataque in ataques:
            # Pega o tipo e o texto gerado (com proteções caso a IA mande formato errado)
            if isinstance(ataque, dict):
                tipo = ataque.get("tipo", "Desconhecido")
                texto = ataque.get("texto", titles_orig[global_i])
            else:
                tipo = "Desconhecido"
                texto = str(ataque)

            registros_ataque.append({
                "id": global_i,
                "original": titles_orig[global_i],
                "tipo_variacao": tipo,
                "attacked": texto
            })

    time.sleep(2) # Pausa para não estourar limite da API (rate limit)

# Criando o DataFrame final
df_attack = pd.DataFrame(registros_ataque)
df_attack.to_csv(ATTACK_PATH, index=False, encoding="utf-8-sig")

print(f"Attack concluído: {len(titles_orig)} títulos geraram {len(df_attack)} linhas → {ATTACK_PATH}")
df_attack.head(15)

Attack: 100%|██████████| 20/20 [08:27<00:00, 25.39s/batch]

Attack concluído: 100 títulos geraram 600 linhas → output/attack.csv


,id,original,tipo_variacao,attacked
0,0,Microscópio Biológico Binocular Meopta Profiss...,Abreviacao,Microscópio Biol. Binoc. Meopta Prof.
1,0,Microscópio Biológico Binocular Meopta Profiss...,Abreviacao,Micro. Biol. Binoc. Meopta Profiss.
2,0,Microscópio Biológico Binocular Meopta Profiss...,Notacao_Unidades,Microscópio Biológico Binocular Meopta Profiss...
3,0,Microscópio Biológico Binocular Meopta Profiss...,Notacao_Unidades,Microscópio Biológico Binocular Meopta Profiss...
4,0,Microscópio Biológico Binocular Meopta Profiss...,Erro_Digitacao,Microscopio Biologcio Binocular Meopta Profiss...
5,0,Microscópio Biológico Binocular Meopta Profiss...,Erro_Digitacao,Microscópio Biológico Binocular Meopta Profiss...
6,1,Nissan Versa,Abreviacao,Niss. Versa
7,1,Nissan Versa,Abreviacao,Niss. Ver.
8,1,Nissan Versa,Notacao_Unidades,Nissan Versa 1 unidade
9,1,Nissan Versa,Notacao_Unidades,Nissan Versa un.


## Passo 3 — Pré-Normalização do Título Atacado (Ditto-inspired)

> Aplica regras simbólicas nos títulos **atacados** antes do LLM de normalização,
> reduzindo variação na entrada (Ditto, Li et al., 2020).

In [19]:
df_attack = pd.read_csv("output/attack.csv")

In [20]:
# ── Pré-normalização do título (Ditto-inspired) ───────────────
# Aplicada ANTES do LLM para reduzir variação na entrada.
# A camada simbólica PÓS-LLM continua existindo como safety net.

_PRE_RULES = [
    # Frações de litro textuais → ml
    (re.compile(r"1/2\s*l(?:itro)?s?\b", re.I),          "500ml"),
    (re.compile(r"meio\s+litro", re.I),                   "500ml"),
    
    # Volume em Litros (agora aceita inteiros e decimais, ex: 0,9 / 1.5 / 2)
    (re.compile(r"(\d+(?:[,.]\d+)?)\s*l(?:itro[s]?)?\b", re.I),
        lambda m: f"{int(float(m.group(1).replace(',', '.')) * 1000)}ml" 
                  if float(m.group(1).replace(',', '.')) * 1000 == int(float(m.group(1).replace(',', '.')) * 1000) 
                  else f"{float(m.group(1).replace(',', '.')) * 1000}ml"),

    # Quilos (aceita decimais também)
    (re.compile(r"(\d+(?:[,.]\d+)?)\s*k(?:ilo)?s?\b", re.I),
        lambda m: f"{m.group(1).replace(' ', '')}kg"),
        
    (re.compile(r"(\d+(?:[,.]\d+)?)\s*(?:gramas?|gr)\b", re.I),
        lambda m: f"{m.group(1).replace(' ', '')}g"),

    # Voltagem
    (re.compile(r"bi[\s-]?volt(?:s)?\b", re.I),          "bivolt"),
    (re.compile(r"(\d+)\s*volt?s?\b", re.I),
        lambda m: f"{m.group(1)}v"),

    # Espaços excessivos
    (re.compile(r"\s{2,}"), " "),
]


def pre_normalize_title(title: str) -> str:
    """Aplica regras de normalização no título ANTES de enviar ao LLM."""
    text = title.strip()
    for pattern, replacement in _PRE_RULES:
        if callable(replacement):
            text = pattern.sub(replacement, text)
        else:
            text = pattern.sub(replacement, text)
    return text.strip()


# ── Teste rápido ──
test_cases = [
    "Maquina De Lavar Electrolux 12 Kilos",
    "Garrafa Térmica 1/2 Litro Aço Inox",
    "Liquidificador Mondial Turbo 110 Volts",
    "Coca Cola meio litro",
    "Panela de Pressão 4,5 Litros",
    "Cerveja Heineken 600 ml Pack 6",
]
print("Pré-normalização (Ditto-inspired):")
print("-" * 65)
for t in test_cases:
    print(f"  {t:<45s} → {pre_normalize_title(t)}")

Pré-normalização (Ditto-inspired):
-----------------------------------------------------------------
  Maquina De Lavar Electrolux 12 Kilos          → Maquina De Lavar Electrolux 12kg
  Garrafa Térmica 1/2 Litro Aço Inox            → Garrafa Térmica 500ml Aço Inox
  Liquidificador Mondial Turbo 110 Volts        → Liquidificador Mondial Turbo 110v
  Coca Cola meio litro                          → Coca Cola 500ml
  Panela de Pressão 4,5 Litros                  → Panela de Pressão 4500ml
  Cerveja Heineken 600 ml Pack 6                → Cerveja Heineken 600 ml Pack 6


In [21]:
df_attack["titulo_pre_normalizado"] = df_attack["attacked"].apply(pre_normalize_title)

alterados = (df_attack["attacked"] != df_attack["titulo_pre_normalizado"]).sum()
print(f"Títulos alterados pela pré-normalização: {alterados}/{len(df_attack)} ({alterados/len(df_attack)*100:.1f}%)")
df_attack[df_attack["attacked"] != df_attack["titulo_pre_normalizado"]][["attacked", "titulo_pre_normalizado"]].head(10)

Títulos alterados pela pré-normalização: 51/600 (8.5%)


,attacked,titulo_pre_normalizado
68,"Essência 0,02L Via Aroma Para Difusor Elétrico",Essência 20ml Via Aroma Para Difusor Elétrico
80,"Cola Adesiva Multiuso Y7000 0,11L Display Celu...",Cola Adesiva Multiuso Y7000 110ml Display Celu...
128,Conjunto Placa Y Disco Sachs Chevrolet Meriva ...,Conjunto Placa Y Disco Sachs Chevrolet Meriva ...
146,"Garrafa Antiga Minuano Limão 0,9 L - Âmbar - Bm",Garrafa Antiga Minuano Limão 900ml - Âmbar - Bm
156,Favo De Mel Flor. Silvestre 200 Gramas,Favo De Mel Flor. Silvestre 200g
157,Favo De Mel Flor. Silv. 200 Gramas,Favo De Mel Flor. Silv. 200g
160,Favo De Mel Florada Silvestree 200 Gramas,Favo De Mel Florada Silvestree 200g
161,Favo De Mel Florda Silvestre 200 Gramas,Favo De Mel Florda Silvestre 200g
200,"Ap Tp-link Wifi Powerline 4220 Av-0,5k -24hs-","Ap Tp-link Wifi Powerline 4220 Av-0,5kg -24hs-"
216,Bomba D Agua Skf VW 22210 8.3 L Diesel,Bomba D Agua Skf VW 22210 8300ml Diesel


## Passo 4 — Normalização via LLM (Batch)

### Regras de Matching Semânticas (Peeters & Bizer, 2023)
> Regras em linguagem natural no prompt alcançaram 88.29% F1 (+5.94%).

### Batch Prompting (BATCHER, Fan et al., 2024)
> 4-7× redução de custo de API com qualidade equivalente.

In [22]:
SYSTEM_PROMPT = """
Você é um especialista em extração de atributos de produtos de e-commerce brasileiro.

══════════════════════════════════════════════════════════════════
TAREFA
══════════════════════════════════════════════════════════════════
Você receberá um ou mais títulos de produtos, cada um identificado por um
índice numérico. Para CADA título, retorne um objeto JSON com as chaves
abaixo. Retorne um ARRAY JSON (lista) contendo um objeto por título, na
mesma ordem recebida. Se receber apenas 1 título, retorne um array com 1
objeto.

Chaves obrigatórias (use null se o atributo não existir no título):
  "idx"                : int    — o índice recebido
  "marca"              : string — ex: "Electrolux", "Samsung", "Tramontina"
  "modelo"             : string — ex: "LTD12", "Galaxy S21", null
  "tipo_produto"       : string — ex: "máquina de lavar", "geladeira"
  "capacidade_volume"  : string — ex: "12kg", "500ml", "2000ml", null
  "voltagem"           : string — "110v", "220v" ou "bivolt"; null se ausente
  "cor"                : string — ex: "branco", "preto", null
  "material"           : string — ex: "inox", "plástico", null
  "quantidade"         : string — ex: "kit 3 peças", "par", null
  "outros_atributos"   : string — outros atributos relevantes, null se não houver

══════════════════════════════════════════════════════════════════
REGRAS DE NORMALIZAÇÃO (unidades e formato)
══════════════════════════════════════════════════════════════════

1. Volume → converter SEMPRE para ml sem espaço:
   "1/2 litro", "meio litro", "0,5 litro" → "500ml"
   "1 litro", "1l" → "1000ml"   |   "1,5 litro" → "1500ml"
   "2 litros" → "2000ml"

2. Voltagem → formato padronizado:
   "110 v", "110 volts", "110V" → "110v"
   "220 v", "220 volts" → "220v"
   "bivolt", "bi-volt", "Bi Volt" → "bivolt"

3. Peso/capacidade → sem espaço:
   "12 kilos", "12 kg", "12KG" → "12kg"
   "500 g", "500gr", "500 gramas" → "500g"

4. Texto em minúsculo para todos os valores, exceto marcas (1ª letra maiúscula).

══════════════════════════════════════════════════════════════════
REGRAS DE DESAMBIGUAÇÃO SEMÂNTICA
══════════════════════════════════════════════════════════════════

5. SINÔNIMOS DE TIPO — use sempre a forma CANÔNICA:
   "lavadora" = "máquina de lavar" → use "máquina de lavar"
   "geladeira" = "refrigerador" → use "geladeira"
   "liq." = "liquidificador" → use "liquidificador"
   "TV" = "televisão" = "televisor" → use "televisão"
   "notebook" = "laptop" → use "notebook"
   "celular" = "smartphone" = "telefone celular" → use "celular"
   "fone" = "fone de ouvido" = "headphone" = "earphone" → use "fone de ouvido"

6. MARCA — unifique variações gráficas da mesma marca:
   "Coca-Cola" = "Coca Cola" = "CocaCola" = "coca cola" → "Coca-Cola"
   Sempre capitalize a 1ª letra de cada palavra da marca.

7. Se o título contém apenas nome genérico sem marca, use null para marca.
   NÃO invente marcas.

8. Se um atributo é ambíguo, prefira a interpretação mais provável para
   o contexto do produto.


══════════════════════════════════════════════════════════════════
FORMATO DE SAÍDA (BATCH)
══════════════════════════════════════════════════════════════════

Retorne SOMENTE um array JSON válido. Sem texto extra, sem markdown.
Cada objeto deve conter o campo "idx" com o índice recebido.
"""

### Função de extração — modo batch

A função agora recebe uma lista de títulos e retorna uma lista de JSONs,
enviando todos em uma única chamada de API.

In [23]:
def extract_batch_llm(
    titles: list[str],
    categories: list[str],
    client: genai.GenerativeModel,
    start_idx: int = 0,
) -> list[dict]:
    """
    Envia múltiplos títulos em uma chamada (BATCHER-inspired).
    Inclui categoria como contexto (Narayan et al., 2022 — attribute selection).
    """
    lines = []
    for i, (title, cat) in enumerate(zip(titles, categories)):
        lines.append(f'[{start_idx + i}] Categoria: "{cat}" | Título: "{title}"')
    user_prompt = "\n".join(lines)

    try:
        response = client.generate_content(user_prompt)
        usage = response.usage_metadata
        raw_text = response.text.strip()

        # Limpa markdown se houver
        if raw_text.startswith("```"):
            raw_text = re.sub(r"^```(?:json)?", "", raw_text).rstrip("`").strip()

        parsed = json.loads(raw_text)

        # Se o LLM retornou um dict único em vez de array, embrulhar
        if isinstance(parsed, dict):
            parsed = [parsed]

        return parsed

    except json.JSONDecodeError:
        # Fallback: tentar extrair cada {...} individualmente
        matches = re.findall(r"\{[^{}]+\}", raw_text, re.DOTALL)
        results = []
        for m in matches:
            try:
                results.append(json.loads(m))
            except json.JSONDecodeError:
                continue
        if results:
            return results

        logger.warning("Sem JSON extraível do batch. Texto: %s", repr(raw_text[:200]))
        return [{} for _ in titles]

    except ValueError as e:
        logger.warning("Resposta bloqueada para batch: %s", e)
        return [{} for _ in titles]

    except Exception as e:
        logger.error("Erro LLM batch: %s", e)
        return [{} for _ in titles]

## Passo 5 — Limpeza Determinística (Camada Simbólica)

> Mantida como safety net pós-LLM.

In [24]:
_RE_SPACES     = re.compile(r"\s+")
_RE_VOLUME_ML  = re.compile(r"(\d+(?:[.,]\d+)?)\s*ml", re.I)
_RE_VOLUME_L   = re.compile(r"(\d+(?:[.,]\d+)?)\s*l(?:itro[s]?)?(?=\b|$)", re.I)
_RE_WEIGHT_KG  = re.compile(r"(\d+(?:[.,]\d+)?)\s*k(?:g|ilos?|ilogramas?)", re.I)
_RE_WEIGHT_G   = re.compile(r"(\d+(?:[.,]\d+)?)\s*g(?:r|ramas?)?(?=\b|$)", re.I)
_RE_VOLTAGE    = re.compile(r"(\d+)\s*v(?:olts?)?(?=\b|$)", re.I)
_RE_BIVOLT     = re.compile(r"bi[\s-]?volt", re.I)


def _clean_str(value: str | None) -> str:
    if not value or not isinstance(value, str):
        return ""
    return _RE_SPACES.sub(" ", value.strip()).lower()


def _normalize_volume(value: str) -> str:
    if not value:
        return value
    value = value.replace(",", ".")
    m = _RE_VOLUME_ML.search(value)
    if m:
        qty = float(m.group(1))
        return f"{int(qty)}ml" if qty == int(qty) else f"{qty}ml"
    m = _RE_VOLUME_L.search(value)
    if m:
        return f"{int(float(m.group(1)) * 1000)}ml"
    return value


def _normalize_weight(value: str) -> str:
    if not value:
        return value
    value = value.replace(",", ".")
    m = _RE_WEIGHT_KG.search(value)
    if m:
        qty = float(m.group(1))
        return f"{int(qty)}kg" if qty == int(qty) else f"{qty}kg"
    m = _RE_WEIGHT_G.search(value)
    if m:
        qty = float(m.group(1))
        return f"{int(qty)}g" if qty == int(qty) else f"{qty}g"
    return value


def _normalize_voltage(value: str) -> str:
    if not value:
        return value
    if _RE_BIVOLT.search(value):
        return "bivolt"
    m = _RE_VOLTAGE.search(value)
    return f"{m.group(1)}v" if m else value


def _normalize_capacity(value: str) -> str:
    if not value:
        return value
    v = value.lower()
    if any(u in v for u in ["ml", "litro", " l", "l "]):
        return _normalize_volume(v)
    if any(u in v for u in ["kg", "kilo", "grama", " g", "g "]):
        return _normalize_weight(v)
    return _clean_str(value)


def clean_llm_output(raw_json: dict) -> dict:
    if not raw_json:
        return {}
    cleaned = {}
    for key, value in raw_json.items():
        if key == "idx":
            continue
        if value is None or value == "null":
            cleaned[key] = None
            continue
        s = str(value).strip()
        if key == "marca":
            cleaned[key] = _RE_SPACES.sub(" ", s).title() or None
        elif key == "voltagem":
            cleaned[key] = _normalize_voltage(_clean_str(s)) or None
        elif key == "capacidade_volume":
            cleaned[key] = _normalize_capacity(_clean_str(s)) or None
        else:
            cleaned[key] = _clean_str(s) or None
    return cleaned


def build_canonical_name(cleaned_json: dict) -> str:
    """Formato: ${nome} ${marca} ${cor} ${outros} - ${quantidade} ${unidade}"""
    nome  = cleaned_json.get("tipo_produto") or ""
    marca = cleaned_json.get("marca") or ""
    cor   = cleaned_json.get("cor") or ""

    outros_parts = [
        cleaned_json.get("modelo"),
        cleaned_json.get("material"),
        cleaned_json.get("voltagem"),
        cleaned_json.get("outros_atributos"),
    ]
    outros = " ".join(p for p in outros_parts if p)

    cap = cleaned_json.get("capacidade_volume") or ""
    m_cap = re.match(r"(\d+(?:\.\d+)?)\s*([a-zA-Z]+)", cap)
    qty_str = f"{m_cap.group(1)} {m_cap.group(2)}" if m_cap else cap

    main = " ".join(p for p in [nome, marca, cor, outros] if p)
    if qty_str:
        return f"{main} - {qty_str}".strip()
    return main

## Passo 6 — Orquestração e Exportação

In [25]:
os.makedirs("output", exist_ok=True)

genai.configure(api_key=API_KEY)
client = genai.GenerativeModel(
    model_name=LLM_MODEL,
    system_instruction=SYSTEM_PROMPT,
    generation_config=genai.GenerationConfig(
        response_mime_type="application/json",
        temperature=TEMPERATURE,
        max_output_tokens=2048,
    ),
)

In [26]:
# ── Loop principal de normalização (batch) ─────────────────────
titles_pre = df_attack["titulo_pre_normalizado"].tolist()
n = len(titles_pre)

all_raw_jsons  = [None] * n
all_cleaned    = [None] * n
all_canonical  = [None] * n

for batch_start in tqdm(range(0, n, BATCH_SIZE), desc="Normalizando", unit="batch"):
    batch_end    = min(batch_start + BATCH_SIZE, n)
    batch_titles = titles_pre[batch_start:batch_end]
    batch_cats   = [""] * len(batch_titles)  # sem categoria no novo fluxo

    raw_list = extract_batch_llm(batch_titles, batch_cats, client, start_idx=batch_start)

    for local_i in range(batch_end - batch_start):
        global_i = batch_start + local_i

        matched = None
        for r in raw_list:
            if isinstance(r, dict) and r.get("idx") == global_i:
                matched = r
                break
        if matched is None and local_i < len(raw_list):
            matched = raw_list[local_i] if isinstance(raw_list[local_i], dict) else {}
        if matched is None:
            matched = {}

        cleaned = clean_llm_output(matched)
        canon   = build_canonical_name(cleaned)

        all_raw_jsons[global_i] = matched
        all_cleaned[global_i]   = cleaned
        all_canonical[global_i] = canon

    time.sleep(2)

print(f"Normalização concluída: {n} títulos em {(n + BATCH_SIZE - 1) // BATCH_SIZE} batches")

Normalizando: 100%|██████████| 120/120 [52:48<00:00, 26.41s/batch] 

Normalização concluída: 600 títulos em 120 batches


In [27]:
df_result = pd.DataFrame({
    "original":   df_attack["original"].values,
    "attacked":   df_attack["attacked"].values,
    "normalized": all_canonical,
})

df_result.to_csv(NORMALIZED_PATH, index=False, encoding="utf-8-sig")
print(f"Exportado: {NORMALIZED_PATH}")
df_result.head(10)

Exportado: output/normalized.csv


,original,attacked,normalized
0,Microscópio Biológico Binocular Meopta Profiss...,Microsc. Biol. Binoc. Meopta Prof.,"microscópio Meopta biológico, binocular, profi..."
1,Microscópio Biológico Binocular Meopta Profiss...,Microscópio Biol. Binocular Meopta Profiss.,"microscópio Meopta biológico, binocular, profi..."
2,Microscópio Biológico Binocular Meopta Profiss...,Microscópio Biológico Binocular Meopta Profiss...,"microscópio Meopta biológico, binocular, profi..."
3,Microscópio Biológico Binocular Meopta Profiss...,Microscópio Biológico Binocular Meopta Profiss...,"microscópio Meopta biológico, binocular, profi..."
4,Microscópio Biológico Binocular Meopta Profiss...,Microscópio Biológioc Binocular Meopta Profiss...,"microscópio Meopta biológico, binocular, profi..."
5,Microscópio Biológico Binocular Meopta Profiss...,Microscópio Biológico Binoculra Meopta Profiss...,microscópio Meopta biológico binocular profiss...
6,Nissan Versa,Niss. Versa,carro Nissan versa
7,Nissan Versa,Niss. Versa,carro Nissan versa
8,Nissan Versa,Nissan Versa 0km,carro Nissan versa 0km
9,Nissan Versa,Nissan Versa Sedan,carro Nissan versa sedan


## Passo 6 — Avaliação Estruturada

> Módulo `avaliacao_pipeline.py` rodado standalone no repositório.
> Aqui importamos e executamos diretamente sobre `df_result`.

In [ ]:
df_result = pd.read_csv("output/normalized.csv")

In [8]:
from eval import gerar_relatorio_robusto

relatorio = gerar_relatorio_robusto(df_result)

  RELATÓRIO DE ROBUSTEZ (ATTACK vs NORMALIZED)
Total de variações processadas: 600
Total de produtos originais únicos: 100
Produtos normalizados perfeitamente (100% resiliente as variações): 14
Taxa de resiliência: 14.0%

  🚨 EXEMPLOS DE QUEBRA DE ROBUSTEZ
Modelos originais listados abaixo geraram resultados canônicos diferentes:

Produto Original: ' Corona Catalano Zanella Rx200 P428 43d '
Variações atacadas e o que o modelo entendeu delas:
  • Atacado:  Corona Catalano Zanella Rx200 P428 43d              → Normalizou: corona Zanella rx200 p428 43d, catalano
  • Atacado:  Corona Catalano Zanella Rx200 P428 43d              → Normalizou: corona Zanella rx200 p428 43d, catalano
  • Atacado:  Corona Catalano Zanella Rx200 P428 43d              → Normalizou: corona Zanella rx200 p428 43d, catalano
  • Atacado:  Corona Catalano Zanella Rx200 P428 43d              → Normalizou: corona Zanella rx200 p428 43d, catalano
  • Atacado:  Corona Catalano Zanella Rx200 P428 43d              → Normal

## Verificação Rápida — Exemplos do Pipeline v2

In [9]:
test_titles = [
    ("COCA COLA 500 ml",                         "Refrigerantes"),
    ("coca-cola 500ml",                           "Refrigerantes"),
    ("Coca Cola 0,5 litro",                       "Refrigerantes"),
    ("CocaCola 1/2 litro",                        "Refrigerantes"),
    ("Coca Cola meio litro",                      "Refrigerantes"),
    ("Lavadora Electrolux 12 Kilos 220v Branca",  "Eletrodomésticos"),
    ("Lavadora Electroluux 12 Kg 220volts Branca",  "Eletrodomésticos"),
    ("Lavadora Electroluux 12 Kilos 220v branca",  "Eletrodomésticos"),
    ("Maquina De Lavar Electrolux 12kg 220 Volts","Eletrodomésticos"),
    ("Liq. Mondial Turbo Inox 1000w 110v",        "Eletrodomésticos"),
    ("Liq. Mondial Turbo Inox 1000w 110volts",        "Eletrodomésticos"),
    ("Kit 3 Pçs Panela Tramontina Inox",          "Cozinha"),
    ("Smart TV Samsung 55 4K Crystal Bivolt",     "Eletrônicos"),
]

# Pré-normalizar
pre_normed = [(pre_normalize_title(t), c) for t, c in test_titles]

print("Pré-normalização:")
print("-" * 65)
for (orig, _), (normed, _) in zip(test_titles, pre_normed):
    flag = " ←" if orig != normed else ""
    print(f"  {orig:<50s} → {normed}{flag}")

# Processar via LLM (batch)
titles_batch = [t for t, _ in pre_normed]
cats_batch   = [c for _, c in pre_normed]
results = extract_batch_llm(titles_batch, cats_batch, client, start_idx=0)

print()
print("Resultados:")
print("-" * 65)
for i, (orig_title, _) in enumerate(test_titles):
    raw = results[i] if i < len(results) else {}
    cleaned = clean_llm_output(raw)
    canon   = build_canonical_name(cleaned)
    print(f"  Título:    {orig_title}")
    print(f"  Canônico:  {canon}")
    print()

NameError: name 'pre_normalize_title' is not defined

In [ ]:
with open("resultados_teste.txt", "w", encoding="utf-8") as f:
    for i, (orig_title, _) in enumerate(test_titles):
        raw = results[i] if i < len(results) else {}
        cleaned = clean_llm_output(raw)
        canon   = build_canonical_name(cleaned)
        f.write(f"Título:    {orig_title}\n")
        f.write(f"Canônico:  {canon}\n\n")

In [ ]:
df_teste_dict = {
    "titulo_original": [],
    "titulo_pre_norm": [], 
    "categoria_original": [],
    "json_llm_bruto": [],  
    "json_limpo_python": [],
    "nome_canonico_final": []
}


for i, (orig_title, category) in enumerate(test_titles):
    raw = results[i] if i < len(results) else {}
    cleaned = clean_llm_output(raw)
    canon   = build_canonical_name(cleaned)
    pre_normed_title = pre_normalize_title(orig_title)
    
    df_teste_dict["titulo_original"].append(orig_title)
    df_teste_dict["titulo_pre_norm"].append(pre_normed_title)
    df_teste_dict["categoria_original"].append(category)
    df_teste_dict["json_llm_bruto"].append(json.dumps(raw, ensure_ascii=False))
    df_teste_dict["json_limpo_python"].append(json.dumps(cleaned, ensure_ascii=False))
    df_teste_dict["nome_canonico_final"].append(canon)

    


In [ ]:
df_teste = pd.DataFrame(df_teste_dict)

In [14]:
df_teste

,titulo_original,titulo_pre_norm,categoria_original,json_llm_bruto,json_limpo_python,nome_canonico_final
0,COCA COLA 500 ml,COCA COLA 500 ml,Refrigerantes,"{""idx"": 0, ""marca"": ""Coca-Cola"", ""modelo"": nul...","{""marca"": ""Coca-Cola"", ""modelo"": null, ""tipo_p...",refrigerante | Coca-Cola | 500ml
1,coca-cola 500ml,coca-cola 500ml,Refrigerantes,"{""idx"": 1, ""marca"": ""Coca-Cola"", ""modelo"": nul...","{""marca"": ""Coca-Cola"", ""modelo"": null, ""tipo_p...",refrigerante | Coca-Cola | 500ml
2,"Coca Cola 0,5 litro",Coca Cola 500ml,Refrigerantes,"{""idx"": 2, ""marca"": ""Coca-Cola"", ""modelo"": nul...","{""marca"": ""Coca-Cola"", ""modelo"": null, ""tipo_p...",refrigerante | Coca-Cola | 500ml
3,CocaCola 1/2 litro,CocaCola 500ml,Refrigerantes,"{""idx"": 3, ""marca"": ""Coca-Cola"", ""modelo"": nul...","{""marca"": ""Coca-Cola"", ""modelo"": null, ""tipo_p...",refrigerante | Coca-Cola | 500ml
4,Coca Cola meio litro,Coca Cola 500ml,Refrigerantes,"{""idx"": 4, ""marca"": ""Coca-Cola"", ""modelo"": nul...","{""marca"": ""Coca-Cola"", ""modelo"": null, ""tipo_p...",refrigerante | Coca-Cola | 500ml
5,Lavadora Electrolux 12 Kilos 220v Branca,Lavadora Electrolux 12kg 220v Branca,Eletrodomésticos,"{""idx"": 5, ""marca"": ""Electrolux"", ""modelo"": nu...","{""marca"": ""Electrolux"", ""modelo"": null, ""tipo_...",máquina de lavar | Electrolux | 12kg | 220v | ...
6,Lavadora Electroluux 12 Kg 220volts Branca,Lavadora Electroluux 12 Kg 220v Branca,Eletrodomésticos,"{""idx"": 6, ""marca"": ""Electrolux"", ""modelo"": nu...","{""marca"": ""Electrolux"", ""modelo"": null, ""tipo_...",máquina de lavar | Electrolux | 12kg | 220v | ...
7,Lavadora Electroluux 12 Kilos 220v branca,Lavadora Electroluux 12kg 220v branca,Eletrodomésticos,"{""idx"": 7, ""marca"": ""Electrolux"", ""modelo"": nu...","{""marca"": ""Electrolux"", ""modelo"": null, ""tipo_...",máquina de lavar | Electrolux | 12kg | 220v | ...
8,Maquina De Lavar Electrolux 12kg 220 Volts,Maquina De Lavar Electrolux 12kg 220v,Eletrodomésticos,"{""idx"": 8, ""marca"": ""Electrolux"", ""modelo"": nu...","{""marca"": ""Electrolux"", ""modelo"": null, ""tipo_...",máquina de lavar | Electrolux | 12kg | 220v
9,Liq. Mondial Turbo Inox 1000w 110v,Liq. Mondial Turbo Inox 1000w 110v,Eletrodomésticos,"{""idx"": 9, ""marca"": ""Mondial"", ""modelo"": ""turb...","{""marca"": ""Mondial"", ""modelo"": ""turbo"", ""tipo_...",liquidificador | Mondial | turbo | 110v | inox...


In [16]:
relatorio = gerar_relatorio(df_teste, verbose=True)

  1. QUALIDADE DA EXTRAÇÃO JSON
  Total de registros:        13
  JSONs válidos com dados:   13  (100.0%)
  JSONs vazios/erro:         0
  JSONs todos nulos:         0
  Taxa de falha total:       0.0%

  2. COMPLETUDE DE ATRIBUTOS
  marca                  ████████████████████  100.0%  (13/13)
  tipo_produto           ████████████████████  100.0%  (13/13)
  capacidade_volume      █████████████░░░░░░░   69.2%  (9/13)
  voltagem               ██████████░░░░░░░░░░   53.9%  (7/13)
  modelo                 ████░░░░░░░░░░░░░░░░   23.1%  (3/13)
  cor                    ████░░░░░░░░░░░░░░░░   23.1%  (3/13)
  material               ████░░░░░░░░░░░░░░░░   23.1%  (3/13)
  outros_atributos       ████░░░░░░░░░░░░░░░░   23.1%  (3/13)
  quantidade             █░░░░░░░░░░░░░░░░░░░    7.7%  (1/13)

  3. ANÁLISE DE AGRUPAMENTO
  Registros com nome:        13
  Nomes canônicos únicos:    6
  Taxa de redução:           53.85%
  Singletons:                3  (50.0% dos nomes)
  Grupos com 2+ membros:     3

In [17]:
with open("eval.py", "r", encoding="utf-8") as f:
    eval_code = f.read()

In [20]:
with open("relatorio_completo.txt", "w", encoding="utf-8") as f:
    original_stdout = sys.stdout
    sys.stdout = f
    try:
        # A função gerar_relatorio agora será a versão sem os cortes
        relatorio = gerar_relatorio(df_teste, verbose=True)
    finally:
        sys.stdout = original_stdout # Restaura o print normal

print("Relatório completo salvo sem cortes no arquivo 'relatorio_completo.txt'")

Relatório completo salvo sem cortes no arquivo 'relatorio_completo.txt'


#### gasto a cada 10 itens: US$ 0,0018315

#### gasto a cada 10 milhoes de itens : US$ 1.831,50.